# [3장 2강] 실습: 분류 평가지표

## 실습 목표

- 고객 이탈 여부를 예측하는 이진 분류 문제를 정의할 수 있다.
- 수치형·범주형 Feature를 분류 모델에 맞게 전처리할 수 있다.
- Logistic Regression으로 고객 이탈 여부와 확률을 예측할 수 있다.
- Confusion Matrix의 TN, FP, FN, TP를 구분할 수 있다.
- Accuracy, Precision, Recall, F1-score와 AUC의 차이를 설명할 수 있다.
- 불균형 데이터에서 Accuracy만으로 평가하면 안 되는 이유를 설명할 수 있다.
- Threshold 변화에 따른 Precision–Recall Trade-off를 해석할 수 있다.

## 사용 데이터

- 파일: `Telco-Customer.csv`
- Label: `Churn` (`No`: 유지, `Yes`: 이탈)
- Positive Class: 이탈 고객 `Yes=1`

## 진행 방식

- 문제 설명과 요구사항을 확인한 뒤 코드 셀을 작성합니다.
- 결과값을 근거로 문제에 제시된 질문에 답합니다.

## 실습 준비: 데이터 불러오기

Google Colab에서는 파일을 Google Drive의 `MyDrive`에 저장합니다. Jupyter Notebook에서는 노트북과 같은 폴더에 저장합니다.

In [4]:
import pandas as pd
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path='/content/drive/MyDrive/Telco-Customer.csv'
except ModuleNotFoundError:
    file_path='Telco-Customer.csv'

telco_df=pd.read_csv(file_path)
display(telco_df.head())
print('데이터 크기:',telco_df.shape)
telco_df.info()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


데이터 크기: (7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   

## 필수 1: 고객 이탈 분류를 위한 데이터 준비

### 문제 1-1: Label 변환과 전처리 Pipeline 구성하기

#### 문제 설명

통신사는 고객 정보와 서비스 이용 정보를 이용해 이탈 여부를 예측하려고 합니다. 식별자를 제거하고, 숫자로 저장되지 않은 `TotalCharges`를 변환한 뒤 수치형·범주형 Feature를 모델이 학습할 수 있도록 전처리합니다.

#### 요구사항

1. `Churn`의 클래스별 개수와 비율을 출력합니다.
2. `customerID`를 입력 Feature에서 제거합니다.
3. `TotalCharges`를 숫자로 변환하고 변환 후 결측치 수를 출력합니다.
4. `Churn`에서 `Yes=1`, `No=0`으로 변환합니다.
5. X와 y를 분리하고 `random_state=42`, `stratify=y`로 학습 80%, 테스트 20%로 나눕니다.
6. 수치형은 중앙값 대체 후 MinMaxScaler, 범주형은 OneHotEncoder를 적용하는 ColumnTransformer를 만듭니다.
7. 전처리기를 학습 데이터에 `fit_transform()`, 테스트 데이터에 `transform()`합니다.
8. 데이터 크기, Label 분포, 변환 후 Feature 개수와 수치형 값의 범위를 출력합니다.

#### 출력 결과

- 유지 5,174명, 이탈 1,869명
- `TotalCharges` 숫자 변환 후 결측치 11개
- 학습 데이터 5,634행, 테스트 데이터 1,409행
- 전처리 후 Feature 30개

#### 결과 해석 작성

> `stratify=y`를 사용한 이유와 결측치 대체·Scaling·Encoding을 학습 데이터에서 먼저 학습해야 하는 이유는 무엇인가요?

In [5]:
# 1. Churn의 클래스별 개수와 비율 출력
print(telco_df['Churn'].value_counts())
print(telco_df['Churn'].value_counts(normalize=True))

# 2. customerID 제거
telco_df = telco_df.drop(columns=['customerID'])


# 3. TotalCharges를 숫자로 변환하고 결측치 수 출력

telco_df['TotalCharges'] = pd.to_numeric(telco_df['TotalCharges'],errors='coerce')
print("TotalCharges 변환 후 결측치 수: ", telco_df['TotalCharges'].isna().sum())


# 4. Churn 변환
# Yes = 1, No = 0
telco_df['Churn'] = telco_df['Churn'].map({'Yes': 1,'No': 0})


# 5. X와 y 분리
# 학습 80%, 테스트 20%
# random_state=42, stratify=y

X = telco_df.drop(columns=['Churn'])
y = telco_df['Churn']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


# 6. ColumnTransformer 생성
# 수치형: 중앙값 대체 → MinMaxScaler
# 범주형: OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])
categorical_pipeline = Pipeline([
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])


# 7. 전처리
# Train → fit_transform
# Test → transform

X_train_transformed = preprocessor.fit_transform(X_train)

X_test_transformed = preprocessor.transform(X_test)


# 8. 결과 출력

print('\n=== 데이터 크기 ===')
print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)


print('\n=== Label 분포 ===')

print('Train:')
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print('\nTest:')
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))


print('\n=== 변환 후 Feature 개수 ===')
print('X_train_transformed:', X_train_transformed.shape)
print('X_test_transformed :', X_test_transformed.shape)
print('Feature 개수:', X_train_transformed.shape[1])


# 수치형 변수의 변환 후 범위 확인

X_train_numeric = preprocessor.named_transformers_['num'].transform(
    X_train[numeric_features]
)

print('\n=== 수치형 값의 범위 ===')
print('최소값:', X_train_numeric.min())
print('최대값:', X_train_numeric.max())

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64
TotalCharges 변환 후 결측치 수:  11

=== 데이터 크기 ===
X_train: (5634, 19)
X_test : (1409, 19)
y_train: (5634,)
y_test : (1409,)

=== Label 분포 ===
Train:
Churn
0    4139
1    1495
Name: count, dtype: int64
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Test:
Churn
0    1035
1     374
Name: count, dtype: int64
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64

=== 변환 후 Feature 개수 ===
X_train_transformed: (5634, 45)
X_test_transformed : (1409, 45)
Feature 개수: 45

=== 수치형 값의 범위 ===
최소값: 0.0
최대값: 1.0000000000000002


C:\Users\rkd76\AppData\Local\Temp\ipykernel_23128\1772879617.py:42: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns


## 필수 2: Logistic Regression 학습 및 평가

### 문제 2-1: Confusion Matrix와 분류 평가지표 확인하기

#### 문제 설명

Logistic Regression으로 고객 이탈 확률과 클래스를 예측합니다. 실제 이탈 고객을 유지 고객으로 잘못 판단하는 FN을 포함해 여러 지표를 함께 확인합니다.

#### 요구사항

1. `LogisticRegression(max_iter=1000)`을 학습합니다.
2. 테스트 데이터의 클래스와 이탈 확률을 예측합니다.
3. Confusion Matrix와 TN, FP, FN, TP를 출력합니다.
4. Accuracy, Precision, Recall, F1-score와 AUC를 계산합니다.
5. 실제값, 예측값과 이탈 확률 앞 10개를 비교합니다.
6. 고객 이탈 문제에서 Accuracy만으로 모델을 평가하면 안 되는 이유를 설명합니다.
7. FN의 의미와 이탈 방지 목적에서 Recall이 중요한 이유를 설명합니다.

#### 결과 해석 작성

> 이 고객 이탈 데이터에서 Accuracy만으로 모델을 평가하면 안 되는 이유는 무엇이며, FN은 어떤 고객을 의미하나요?

## 필수 3: 평가지표 직접 계산 및 기준 모델 비교

### 문제 3-1: Confusion Matrix 공식과 다수 클래스 기준 모델 확인하기

#### 문제 설명

혼동행렬의 네 값으로 Accuracy, Precision, Recall과 F1-score를 직접 계산합니다. 또한 모든 고객을 다수 클래스인 유지 고객으로 예측하는 기준 모델을 만들어 Accuracy가 높더라도 이탈 고객을 찾지 못할 수 있음을 확인합니다.

#### 요구사항

1. TN, FP, FN, TP로 네 가지 지표를 직접 계산합니다.
2. scikit-learn 결과와 일치하는지 확인합니다.
3. 모든 테스트 고객을 `0`으로 예측하는 기준 모델을 만듭니다.
4. 기준 모델의 Confusion Matrix, Accuracy, Precision, Recall과 F1-score를 출력합니다.
5. 기준 모델의 Accuracy가 70%를 넘는데도 유용한 이탈 예측 모델이 아닌 이유를 설명합니다.

#### 결과 해석 작성

> 모든 고객을 유지 고객으로 예측한 기준 모델의 Accuracy가 약 73.5%인데도 이탈 예측 모델로 사용할 수 없는 이유는 무엇인가요?

## 심화 1: 분류 Threshold 변경과 Recall 비교

### 문제 4-1: Threshold 0.5와 0.3 비교하기

#### 문제 설명

이탈 고객을 놓치는 FN을 줄이기 위해 기본 Threshold 0.5와 더 낮은 0.3을 비교합니다. Threshold를 낮추면 더 많은 고객을 이탈로 판단하므로 Recall과 FP가 함께 변할 수 있습니다.

#### 요구사항

1. 이탈 확률에 Threshold 0.5와 0.3을 각각 적용합니다.
2. 두 Confusion Matrix와 TN, FP, FN, TP를 출력합니다.
3. Accuracy, Precision, Recall과 F1-score 비교표를 만듭니다.
4. Threshold를 낮췄을 때 FP와 FN의 변화를 설명합니다.
5. 이탈 방지 캠페인에서 두 Threshold 중 무엇을 선택할지 비용 관점에서 작성합니다.

#### 결과 해석 작성

> Threshold를 0.5에서 0.3으로 낮추면 FP, FN, Precision과 Recall은 어떻게 변하며, 어떤 상황에서 0.3이 더 적절한가요?

## 실습 마무리

1. Accuracy, Precision, Recall과 F1-score는 각각 어떤 질문에 답하나요?
2. AUC는 무엇을 평가하나요?
3. 고객 이탈 문제에서 Recall과 Precision 중 하나만 항상 우선할 수 없는 이유는 무엇인가요?